Assignment:
1. 
2. 
3. 

In [1]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

%load_ext autoreload
%autoreload 2

sys.path.append("../../")

MARKET_DATA_PATH = "../../market_data/intraday/"

In [2]:
STOCKS = ["AMZN", "GOOG", "MSFT", "GS", "JPM", "MS", "CVX", "XOM"]    # US stocks
TICKERS = ["SPY"] + STOCKS    # all tickers
PERIOD = 5

In [3]:
frames = []
for t in TICKERS:
    fp = f"{MARKET_DATA_PATH}{t}_{PERIOD}mn.csv"
    df = pd.read_csv(fp, index_col=0, parse_dates=True)[["Close"]]
    df.columns = [t]
    frames.append(df)
prices_raw = pd.concat(frames, axis=1).sort_index()

### Basket Covariance

In [4]:
downsample_period = 120

prices = prices_raw.resample(f"{downsample_period}min").last().dropna(how="any")
rets_all = prices.pct_change().dropna(how="any")

# plot correlation matrix (full sample)
fig = px.imshow(rets_all.corr())
fig.show()

In [5]:
cov_matrix = rets_all.cov()*10000*252*6.5*60/120
round(cov_matrix, 1)

,SPY,AMZN,GOOG,MSFT,GS,JPM,MS,CVX,XOM
SPY,205.3,307.3,231.1,211.8,302.3,216.3,291.9,111.7,95.8
AMZN,307.3,919.1,422.5,384.1,404.8,295.2,397.2,138.8,79.4
GOOG,231.1,422.5,767.3,257.5,283.8,190.4,267.4,68.5,33.8
MSFT,211.8,384.1,257.5,519.1,243.8,159.9,231.5,49.8,23.6
GS,302.3,404.8,283.8,243.8,817.1,536.5,718.5,212.7,183.8
JPM,216.3,295.2,190.4,159.9,536.5,556.2,510.3,179.0,154.1
MS,291.9,397.2,267.4,231.5,718.5,510.3,819.4,201.6,174.1
CVX,111.7,138.8,68.5,49.8,212.7,179.0,201.6,401.9,328.9
XOM,95.8,79.4,33.8,23.6,183.8,154.1,174.1,328.9,407.4


In [6]:
# Rolling covariance of US equities (equal-weight universe); returns from pct_change
ROLL_WINDOW = 60  # bars on the resampled grid

rets = rets_all[STOCKS]    # only stocks
n_assets = rets.shape[1]
w = np.ones(n_assets) / n_assets  # equal weights

# Pairwise rolling sample covariance Σ_t for each window end t (MultiIndex: time × asset)
rolling_cov = rets.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).cov()

# Equal-weight basket return; its rolling variance equals w @ Σ_t @ w (same ddof as pandas .var)
basket_rets = rets.mean(axis=1)
rolling_basket_var = basket_rets.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).var(ddof=1)
rolling_index_var = rets_all["SPY"].rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).var(ddof=1)

last_t = rets.index[-1]
cov_last = rolling_cov.loc[last_t]

# plot rolling variance of equal-weight basket (w′Σw) and SPY index
fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_index_var.index, y=rolling_index_var, mode="lines", name="SPY index"))
fig.add_trace(go.Scatter(x=rolling_basket_var.index, y=rolling_basket_var, mode="lines", name="equal-weight basket"))
fig.update_layout(
    title="Rolling variance of equal-weight basket (w′Σw) and SPY index",
    template="plotly_dark"
)
fig.update_layout(xaxis_title=None, yaxis_title="Basket variance", legend_title="Basket")
fig.show()


In [7]:
# rolling correlation (average)
rolling_corr = rets.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).corr()
avg_corr = rolling_corr.groupby(level=0).mean().mean(axis=1)

# plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=avg_corr.index, y=avg_corr, mode="lines", name="Rolling average correlation"))
fig.update_layout(
    title="Rolling average correlation",
    template="plotly_dark"
)
fig.update_layout(xaxis_title=None, yaxis_title="Correlation", legend_title="Correlation")
fig.update_yaxes(tickformat=".0%")
fig.show()


### Ledoit-Wolf shrinkage

In [8]:
# the covariance matrix is full rank, therefore invertibility is not an issue
cov_matrix

,SPY,AMZN,GOOG,MSFT,GS,JPM,MS,CVX,XOM
SPY,205.307462,307.296644,231.076209,211.825477,302.315061,216.275461,291.929524,111.659265,95.766766
AMZN,307.296644,919.131121,422.541117,384.145903,404.821855,295.162544,397.154825,138.822828,79.423171
GOOG,231.076209,422.541117,767.307347,257.471991,283.790123,190.377830,267.407761,68.469111,33.845147
MSFT,211.825477,384.145903,257.471991,519.074046,243.754736,159.888535,231.458152,49.793108,23.639551
GS,302.315061,404.821855,283.790123,243.754736,817.100518,536.486951,718.525064,212.735061,183.767180
JPM,216.275461,295.162544,190.377830,159.888535,536.486951,556.185037,510.294555,178.973290,154.098895
MS,291.929524,397.154825,267.407761,231.458152,718.525064,510.294555,819.413835,201.611706,174.124930
CVX,111.659265,138.822828,68.469111,49.793108,212.735061,178.973290,201.611706,401.853969,328.890489
XOM,95.766766,79.423171,33.845147,23.639551,183.767180,154.098895,174.124930,328.890489,407.390478


In [9]:
# matrix rank
rank = np.linalg.matrix_rank(cov_matrix)
int(rank)


9

In [14]:
# let's make the covariance matrix singular by adding colinear prices
rets_sing = rets_all.copy()
for s in STOCKS:
    rets_sing[f'{s}2'] = rets_sing[s]*2

n_assets2 = rets_sing.shape[1]
# new covariance matrix
cov_sing = rets_sing.cov()*10000*252*6.5*60/120
round(cov_sing, 1)


,SPY,AMZN,GOOG,MSFT,GS,JPM,MS,CVX,XOM,AMZN2,GOOG2,MSFT2,GS2,JPM2,MS2,CVX2,XOM2
SPY,205.3,307.3,231.1,211.8,302.3,216.3,291.9,111.7,95.8,614.6,462.2,423.7,604.6,432.6,583.9,223.3,191.5
AMZN,307.3,919.1,422.5,384.1,404.8,295.2,397.2,138.8,79.4,1838.3,845.1,768.3,809.6,590.3,794.3,277.6,158.8
GOOG,231.1,422.5,767.3,257.5,283.8,190.4,267.4,68.5,33.8,845.1,1534.6,514.9,567.6,380.8,534.8,136.9,67.7
MSFT,211.8,384.1,257.5,519.1,243.8,159.9,231.5,49.8,23.6,768.3,514.9,1038.1,487.5,319.8,462.9,99.6,47.3
GS,302.3,404.8,283.8,243.8,817.1,536.5,718.5,212.7,183.8,809.6,567.6,487.5,1634.2,1073.0,1437.1,425.5,367.5
JPM,216.3,295.2,190.4,159.9,536.5,556.2,510.3,179.0,154.1,590.3,380.8,319.8,1073.0,1112.4,1020.6,357.9,308.2
MS,291.9,397.2,267.4,231.5,718.5,510.3,819.4,201.6,174.1,794.3,534.8,462.9,1437.1,1020.6,1638.8,403.2,348.2
CVX,111.7,138.8,68.5,49.8,212.7,179.0,201.6,401.9,328.9,277.6,136.9,99.6,425.5,357.9,403.2,803.7,657.8
XOM,95.8,79.4,33.8,23.6,183.8,154.1,174.1,328.9,407.4,158.8,67.7,47.3,367.5,308.2,348.2,657.8,814.8
AMZN2,614.6,1838.3,845.1,768.3,809.6,590.3,794.3,277.6,158.8,3676.5,1690.2,1536.6,1619.3,1180.7,1588.6,555.3,317.7


In [24]:
# rank
rank = np.linalg.matrix_rank(cov_sing)

if rank == n_assets2:
    print("The covariance matrix is full rank.")
else:
    print("The covariance matrix is singular.")


The covariance matrix is singular.


In [23]:
# shrinkage applied to covariance matrix
alpha = 0.01
cov_shrink = (1 - alpha) * cov_sing + alpha * np.trace(cov_sing) / n_assets2 * np.eye(n_assets2)
round(cov_shrink, 1)

# rank
rank = np.linalg.matrix_rank(cov_shrink)
if rank == n_assets2:
    print("The shrunk covariance matrix is full rank.")
else:
    print("The shrunk covariance matrix is singular.")


The shrunk covariance matrix is full rank.
